# 📚 Mastering LangChain Text Splitters
## Deep Dive: `RecursiveCharacterTextSplitter` for RAG & LLMs

In **Retrieval-Augmented Generation (RAG)** systems, large documents (PDFs, docs, web pages) cannot be embedded or fed into an LLM all at once. 

### 🎯 Why is Text Chunking Essential?
1. **Embedding Quality:** Dense vector models produce more accurate embeddings for cohesive, bite-sized ideas than for entire multi-page documents.
2. **Context Window Limits:** LLMs have finite token budgets. Smaller, targeted chunks save prompt tokens and reduce hallucination.
3. **Precision Retrieval:** When a user asks a question, vector search retrieves only the relevant paragraphs rather than an entire book.

---

## 📑 Separator Reference Cheat Sheet

| Priority | Separator | Meaning | Granularity | What it Splits | Best Used For |
| :---: | :---: | :---: | :---: | :---: | :--- |
| **1** | `"\n\n"` | Double Newline | Macro | Paragraphs | Keeps complete thoughts, paragraphs, and arguments together |
| **2** | `"\n"` | Single Newline | Medium | Lines / Sentences | Preserves single lines, bullet points, or code blocks |
| **3** | `" "` | Single Space | Fine | Words | Splits sentences while ensuring words are never cut mid-spelling |
| **4** | `""` | Empty String | Micro | Characters | Hard fallback; slices character-by-character (cuts words if needed) |



In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """Python is a programming language.
Python supports OOP.
Python supports multiple libraries.
LangChain is used to build LLM applications.
RAG helps LLMs retrieve external information."""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=60,
    chunk_overlap=0,
    separators=[""]
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print("----------------")

Chunk 1:
Python is a programming language.
Python supports
----------------
Chunk 2:
OOP.
Python supports multiple libraries.
LangChain is used
----------------
Chunk 3:
to build LLM applications.
RAG helps LLMs retrieve external
----------------
Chunk 4:
information.
----------------


# Overlapping in chunking

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """
Artificial Intelligence is transforming the world.
Machine Learning is a subset of Artificial Intelligence.
Deep Learning is a subset of Machine Learning.
Large Language Models use Deep Learning techniques.
"""

# Without overlap
splitter1 = RecursiveCharacterTextSplitter(
    chunk_size=70,
    chunk_overlap=0,
    separators=[" "]
)

chunks1 = splitter1.split_text(text)

print("WITHOUT OVERLAP")

for i, chunk in enumerate(chunks1):
    print(f"Chunk {i+1}: {chunk}")


# With overlap
splitter2 = RecursiveCharacterTextSplitter(
    chunk_size=70,
    chunk_overlap=20,
    separators=[" "]
)

chunks2 = splitter2.split_text(text)

print("\nWITH OVERLAP")

for i, chunk in enumerate(chunks2):
    print(f"Chunk {i+1}: {chunk}")


WITHOUT OVERLAP
Chunk 1: Artificial Intelligence is transforming the world.
Machine Learning
Chunk 2: is a subset of Artificial Intelligence.
Deep Learning is a subset of
Chunk 3: Machine Learning.
Large Language Models use Deep Learning
Chunk 4: techniques.

WITH OVERLAP
Chunk 1: Artificial Intelligence is transforming the world.
Machine Learning
Chunk 2: Learning is a subset of Artificial Intelligence.
Deep Learning is a
Chunk 3: Learning is a subset of Machine Learning.
Large Language Models use
Chunk 4: Language Models use Deep Learning techniques.


# Generate embeddings

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

text = "Artificial Intelligence is transforming the world."

vector = embeddings.embed_query(text)

print(vector)
print("Embedding dimensions:", len(vector))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[0.038724154233932495, -0.0011055220384150743, 0.08271618187427521, -0.016288595274090767, 0.0465431734919548, -0.00953030027449131, -0.029974931851029396, 0.0034941399935632944, 0.011196283623576164, 0.0026302312035113573, -0.013326141983270645, 0.0720948800444603, -0.004090472124516964, 0.04323472082614899, -0.035830553621053696, 0.03188560530543327, -0.10739818215370178, -0.03442884236574173, -0.1473446637392044, -0.08242831379175186, 0.0094416793435812, 0.04448235407471657, -0.01045930851250887, -0.022715236991643906, -0.006941288709640503, 0.05520905926823616, 0.023868948221206665, -0.0628514215350151, -0.02650441788136959, -0.0503157377243042, 0.016014153137803078, 0.061826251447200775, 0.03329753503203392, 0.024846989661455154, -0.02860439196228981, 0.08128844201564789, 0.027321098372340202, -0.014599109999835491, 0.0613124780356884, -0.04106493666768074, 0.031062766909599304, -0.08352870494127274, 0.006512032821774483, -0.021333860233426094, 0.06933078169822693, 0.1365005820989

# what is an embedding?

An embedding is a numerical representation of text.

An embedding model converts human-readable text into a list of numbers (a vector).

For example, consider this sentence: Artificial Intelligence is transforming the world.

The embedding model converts it into something like: [0.12, 0.45, -0.23, 0.67, 0.81, ...] 

These numbers represent learned semantic features of the text.

# What does dimension 384 mean?

Let's take the Hugging Face model: sentence-transformers/all-MiniLM-L6-v2

This model converts each input sentence into a vector containing exactly 384 numerical values.

# Understand dimensions with a simple example?

Imagine you want to represent a person using three characteristics:-

- Height
- Weight
- Age

You can represent a person like this: person = [165, 60, 25]

- Height = 165
- Weight = 60
- Age = 25

How many dimensions do we have?

Answer: 3 dimensions.

# Similarly, an embedding model represents text using 384 numerical values instead of just three characteristics.

sentence = "I love Python programming"

embedding = [
    0.12,
    0.45,
    -0.23,
    0.67,
    # ... more values
]

The actual model learns these representations during training. We cannot directly interpret each individual coordinate as a specific human-readable characteristic.

In [2]:
from sentence_transformers import SentenceTransformer

# Load embedding model
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Input sentences
sentences = [
    "I love Python programming",
    "Machine Learning is a subset of AI",
    "I enjoy developing software"
]

# Generate embeddings
vectors = model.encode(sentences)

# Print embeddings
for i, vector in enumerate(vectors):
    print(f"\nSentence {i+1}")
    print("Embedding:", vector)
    print("Dimensions:", len(vector))

print("\nShape:", vectors.shape)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Sentence 1
Embedding: [-7.95331374e-02  8.64117499e-03 -1.59163233e-02  1.15700494e-02
 -4.87519475e-03 -1.59397259e-01  1.58045404e-02  6.83574229e-02
 -1.69478375e-02  2.50685029e-02 -5.98058477e-02  3.26516964e-02
  6.87324256e-02  5.07102236e-02  5.51671200e-02 -2.33116765e-02
 -9.09842104e-02 -1.05387270e-02  2.70811208e-02 -1.27690285e-01
 -1.17048301e-01  5.44177741e-02 -2.09162524e-03 -8.40950306e-05
  3.44102569e-02 -3.12112155e-04  7.23862136e-03 -2.40813550e-02
 -4.03960049e-03 -6.85073659e-02 -6.82228506e-02  9.75994989e-02
  4.49765176e-02  2.22518090e-02  1.39698340e-02  4.64122668e-02
  3.96304242e-02 -6.00188747e-02 -4.48231921e-02  2.79450379e-02
 -5.21483049e-02  5.17958403e-02 -7.90968351e-03 -4.03999314e-02
 -3.69645581e-02 -1.11603569e-02 -5.53678200e-02 -1.56360101e-02
  1.03882633e-01  2.87843160e-02 -4.29253653e-02 -8.92733224e-03
 -6.39390126e-02 -5.53120710e-02  1.79188568e-02 -1.07116615e-02
  9.47266892e-02 -1.96499862e-02  1.61374640e-02 -1.41081959e-01
 -

# ChromaDB does not simply store your PDF text. It stores the document content, its embedding vectors, metadata, and unique IDs so that we can retrieve relevant information later.

Let's understand step by step.

1. What is ChromaDB?

ChromaDB is a vector database used to store and search numerical embeddings.

In RAG, we use it to store the embeddings generated from our document chunks.

For example, suppose you have these two chunks:

chunks = [
    "Python is a programming language.",
    "Machine Learning is a subset of AI."
]



{
  "ids": [
    "doc_001",
    "doc_002"
  ],

  "documents": [
    "Python is a programming language.",

    "Machine Learning is a subset of AI."
  ],

  "embeddings": [
    [0.12, 0.45, -0.23, 0.67, 0.89],

    [0.34, -0.12, 0.56, 0.78, -0.45]
  ],

  "metadatas": [
    {
      "source": "python.pdf",
      "page": 1
    },
    
    {
      "source": "ml.pdf",
      "page": 2
    }
  ]
}



In [5]:
## The below code will show u how the data is stored in chromaDB

# from langchain_chroma import Chroma
# import json

# vector_store = Chroma.from_documents(
#     documents=chunks,
#     embedding=embeddings,
#     persist_directory="./chroma_db"
# )

# print("Documents stored successfully!")

# # Retrieve stored data from ChromaDB
# stored_data = vector_store.get(
#     include=["documents", "metadatas", "embeddings"]
# )

# # Print IDs
# print("\n========== DOCUMENT IDs ==========")
# print(json.dumps(stored_data["ids"], indent=4))

# # Print original document content
# print("\n========== DOCUMENTS ==========")
# print(json.dumps(stored_data["documents"], indent=4))

# # Print metadata
# print("\n========== METADATA ==========")
# print(json.dumps(stored_data["metadatas"], indent=4))

# # Print embedding vectors
# print("\n========== EMBEDDINGS ==========")
# for i, embedding in enumerate(stored_data["embeddings"]):
#     print(f"\nDocument {i + 1}")
#     print("Embedding dimensions:", len(embedding))
#     print("Embedding vector:", embedding)

In [ ]:
##-- The Below Code Will show how the Retrival is working
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Load Hugging Face embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. Create dummy documents
documents = [
    "Python is a programming language used for AI and machine learning.",
    "Machine learning allows computers to learn from data.",
    "Deep learning uses neural networks to solve complex problems.",
    "FastAPI is a Python framework used to build REST APIs.",
    "ChromaDB is a vector database used to store and retrieve embeddings."
]

# 3. Create ChromaDB
vector_store = Chroma(
    collection_name="student_demo",
    embedding_function=embeddings,
    persist_directory="./chroma_dbsss"
)

# 4. Store documents
ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]

vector_store.add_texts(
    texts=documents,
    ids=ids
)

print("Documents stored successfully!")

# 5. Take user input
query = input("\nEnter your question: ")

# 6. Convert user question into vector
query_embedding = embeddings.embed_query(query)

print("\nQuery converted into embedding!")
print("Embedding dimensions:", len(query_embedding))

# 7. Search in ChromaDB using query vector
results = vector_store.similarity_search_by_vector_with_relevance_scores(
    query_embedding,
    k=3
)

# 8. Display matching documents
print("\n===== SEARCH RESULTS =====")

for i, (doc, score) in enumerate(results):

    print(f"\nResult {i+1}")
    print("Content:", doc.page_content)
    print("Metadata:", doc.metadata)
    print("Relevance Score:", round(score, 4))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documents stored successfully!

Query converted into embedding!
Embedding dimensions: 384

===== SEARCH RESULTS =====

Result 1
Content: Python is a programming language used for AI and machine learning.
Metadata: {}
Relevance Score: 0.2999

Result 2
Content: FastAPI is a Python framework used to build REST APIs.
Metadata: {}
Relevance Score: 1.0354

Result 3
Content: Machine learning allows computers to learn from data.
Metadata: {}
Relevance Score: 1.5657


# Step A: User enters a question

"Vapour Absorption Refrigeration Principle"
# Step B: Convert question into embedding
Embedding Model (Hugging Face)

[0.023, -0.045, 0.078, ...]

# Step C: Similarity Search
ChromaDB compares:
- User question embedding
- Previously stored document embeddings

# Step D: Retrieve matching chunks
Original text chunks are retrieved.
- Chunk 1: Refrigeration principle...
- Chunk 2: Absorption process...
- Chunk 3: Generator working...

# Step E: Pass text to LLM
Prompt = Retrieved Text + User Question

# Step F: LLM generates answer
Final human-readable answer based on the retrieved context.